# Chapter 3 · The transformer, mechanically — Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omar-florez/training-efficient-llms/blob/main/labs/ch03_transformer.ipynb)

Reproduces **Chapter 3**: the 3-token attention lookup of Table 3.1 (then as a heatmap), the from-scratch causal attention module checked against the numpy math, the parameter count of Llama 2 7B to the exact digit, the KV-cache arithmetic, and the sinusoidal position code.

📖 Read the chapter: [The transformer, mechanically](https://omar-florez.github.io/training-efficient-llms/pdf/ch03_transformer.pdf)

In [ ]:
import numpy as np, matplotlib.pyplot as plt

NAVY, BLUE, LIGHT, AMBER, GRAY = "#17406b", "#3f74b8", "#9dbfe4", "#b45309", "#5c5c5c"
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white","axes.edgecolor":GRAY,
    "axes.labelcolor":"#1a1a1a","axes.grid":True,"grid.color":"#e3eaf3","grid.linewidth":0.8,
    "axes.spines.top":False,"axes.spines.right":False,"font.size":11,"figure.dpi":110})
print("ready")

## 1 · Attention by hand: the 3-token lookup

Chapter 3.2's worked example — *The, cat, sat* with d_k = 2 — computed exactly, then the causal-mask weight matrix as a heatmap.

In [ ]:
Q = K = np.array([[1., 0.], [0., 1.], [1., 1.]])
V = np.eye(3)
S = Q @ K.T / np.sqrt(2)
mask = np.triu(np.ones((3, 3), bool), 1)
S_masked = np.where(mask, -np.inf, S)
W = np.exp(S_masked); W /= W.sum(1, keepdims=True)
O = W @ V

print("scores/√2:\n", S.round(2))
print("\ncausal weights:\n", W.round(2))
print("\noutput for 'sat':", O[2].round(2), "  ← 25% The, 25% cat, 50% itself (Table 3.1)")

toks = ["The", "cat", "sat"]
fig, ax = plt.subplots(figsize=(4.2, 3.6))
im = ax.imshow(W, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(3), toks); ax.set_yticks(range(3), toks)
ax.set_xlabel("attends to (key)"); ax.set_ylabel("query")
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{W[i,j]:.2f}", ha="center", va="center",
                color="white" if W[i,j] > 0.5 else NAVY)
ax.set_title("Causal attention weights"); plt.colorbar(im, shrink=0.8); plt.show()

## 2 · The twenty-line module, verified

The chapter's from-scratch `CausalSelfAttention`, run on real tensors, with the mask checked: position i must never receive gradient/signal from j > i. *(Needs PyTorch — preinstalled on Colab.)*

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)

class CausalSelfAttention(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.h, self.dk = n_heads, d // n_heads
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.out = nn.Linear(d, d, bias=False)
    def forward(self, x):
        B, L, d = x.shape
        q, k, v = self.qkv(x).split(d, dim=-1)
        q, k, v = (t.view(B, L, self.h, self.dk).transpose(1, 2) for t in (q, k, v))
        scores = q @ k.transpose(-2, -1) / self.dk ** 0.5
        mask = torch.triu(torch.ones(L, L, dtype=torch.bool), 1)
        w = scores.masked_fill(mask, float("-inf")).softmax(-1)
        return self.out((w @ v).transpose(1, 2).reshape(B, L, d))

attn = CausalSelfAttention(d=64, n_heads=4)
x = torch.randn(2, 10, 64)
print("output shape:", attn(x).shape)

# causality test: perturb a FUTURE token, outputs at earlier positions must not change
y1 = attn(x)
x2 = x.clone(); x2[:, 7] += 100.0        # smash position 7
y2 = attn(x2)
print("positions 0-6 unchanged:", torch.allclose(y1[:, :7], y2[:, :7], atol=1e-5))
print("position  7+  changed  :", not torch.allclose(y1[:, 7:], y2[:, 7:]))

## 3 · Count Llama 2 7B by hand — to the digit

Five integers → the parameter count. The released checkpoint has **6,738,415,616** parameters; our formula lands within the 65 RMSNorm vectors of it.

In [ ]:
def count_params(d, N, d_ff, V, h, h_kv=None, gated=True, tied=False):
    h_kv = h_kv or h; dh = d // h
    attn = 2*d*d + 2*d*(h_kv*dh)
    mlp  = (3 if gated else 2) * d * d_ff
    emb  = V * d * (1 if tied else 2)
    return N*(attn+mlp) + emb, {"embedding": emb, "attention": N*attn, "MLP": N*mlp}

models = {
  "GPT-2 small":  dict(d=768,  N=12, d_ff=3072,  V=50257,  h=12, gated=False, tied=True),
  "Llama 2 7B":   dict(d=4096, N=32, d_ff=11008, V=32000,  h=32),
  "Llama 3 8B":   dict(d=4096, N=32, d_ff=14336, V=128256, h=32, h_kv=8),
}
shares = {}
for name, cfg in models.items():
    total, parts = count_params(**cfg)
    shares[name] = parts
    print(f"{name:>12}: {total/1e9:6.2f} B   ({total:,})")
print("\nLlama 2 7B checkpoint: 6,738,415,616 — difference is 65 RMSNorm vectors × 4096 = 266,240")

fig, ax = plt.subplots(figsize=(9, 3.2))
names = list(shares)
left = np.zeros(len(names))
for comp, color in [("embedding", LIGHT), ("attention", BLUE), ("MLP", NAVY)]:
    vals = np.array([shares[n][comp] for n in names], float)
    tots = np.array([sum(shares[n].values()) for n in names], float)
    frac = vals / tots
    ax.barh(names, frac, left=left, color=color, label=comp)
    for i, f in enumerate(frac):
        if f > 0.07: ax.text(left[i]+f/2, i, f"{f*100:.0f}%", ha="center", va="center", color="white", fontsize=9)
    left += frac
ax.set_xlim(0, 1); ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.25))
ax.set_title("Where the parameters live (Figure 3.10)", pad=28); plt.tight_layout(); plt.show()

## 4 · The KV cache: why serving is a memory problem

`2 · N · h_kv · d_head · bytes` per token. Watch it cross the *weights* line as context grows — and watch GQA move the crossing.

In [ ]:
L = np.arange(0, 33000, 256)
def cache_gb(h_kv, N=32, dh=128, bytes_=2, batch=32):
    return 2*N*h_kv*dh*bytes_ * L * batch / 1e9

plt.figure(figsize=(8.5, 4.2))
plt.plot(L, cache_gb(32), color=NAVY, lw=2, label="MHA 32 KV heads (Llama 2 7B)")
plt.plot(L, cache_gb(8),  color=BLUE, lw=2, label="GQA 8 KV heads (Llama 3 8B)")
plt.plot(L, cache_gb(1),  color=LIGHT, lw=2, label="MQA 1 KV head")
plt.axhline(13.5, color=AMBER, ls="--", lw=1.5); plt.annotate("the model's weights (13.5 GB)", (500, 15), color=AMBER, fontsize=9)
plt.xlabel("context length (batch of 32 sequences)"); plt.ylabel("KV cache, GB")
plt.legend(); plt.title("The cache outgrows the model (Chapter 3.7 → Chapter 13)")
plt.show()
print(f"MHA 7B at 32K, batch 32: {cache_gb(32)[-1]:.0f} GB of cache")

## 5 · The sinusoidal position code (Figure 3.7 as a heatmap)

Fast dimensions distinguish neighbors; slow dimensions distinguish regions — a multi-speed clock, and the frequency ladder RoPE reuses as rotations.

In [ ]:
d_model, positions = 64, 128
pos = np.arange(positions)[:, None]; i = np.arange(d_model)[None, :]
angle = pos / (10000 ** (2*(i//2)/d_model))
PE = np.where(i % 2 == 0, np.sin(angle), np.cos(angle))

plt.figure(figsize=(9, 3.6))
plt.imshow(PE.T, aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
plt.xlabel("position"); plt.ylabel("embedding dimension")
plt.title("Sinusoidal positions: fast rows at the bottom, near-constant rows at the top")
plt.colorbar(shrink=0.8); plt.show()

## Break things

1. In §2, delete the `/ self.dk ** 0.5` and re-run §1's example at d_k = 128 with random Q,K — plot the softmax weights. What happened to them? (This is the most common from-scratch bug.)
2. In §3, price **your own** dream model: pick d, N, V for a Spanish-first 13B and report all four facts.
3. In §4, add DeepSeek-V3's MLA line: 576 elements × 61 layers per token. Where does it sit?
4. In §2, replace the boolean mask with `float("-inf")` in **float16** and find the NaN (Chapter 3.8, pitfall 3).